In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / "data" / "manifests").exists():
    ROOT = Path(r"D:\\FILE AND TASK\\TA")
assert (ROOT / "data" / "manifests").exists(), ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["MPLBACKEND"] = "Agg"

try:
    from IPython.display import display, Image
except ImportError:
    def display(item):
        print(item)

import json
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt

print("Project:", ROOT)
print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Librosa:", librosa.__version__)


Project: D:\FILE AND TASK\TA
Python: c:\Users\Fabio\AppData\Local\Programs\Python\Python314\python.exe
NumPy: 2.5.2
Pandas: 3.0.5
Librosa: 1.0.0


In [2]:
# Kode run_eda_formal.py ditempatkan langsung di notebook ini.
# Entry point CLI tidak dipanggil otomatis agar hasil valid tidak tertimpa.
__file__ = str(ROOT / "scripts" / "run_eda_formal.py")

"""Create a formal, manifest-led EDA inventory without touching the manuscript.

The prior EDA notebook contained output from a former dataset/regime. This
script supersedes it and writes an inspectable physical-audio inventory and
plots to one timestamped reproduction directory.
"""


import hashlib
import json
import os
import subprocess
from datetime import datetime, timezone
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parent.parent
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "results" / ".mplconfig"))
os.environ["MPLBACKEND"] = "Agg"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf

MANIFEST_DIR = PROJECT_ROOT / "data" / "manifests"
OUTPUT_ROOT = PROJECT_ROOT / "results" / "reproductions"
AUDIO_SUFFIXES = {".mp3", ".wav", ".flac", ".ogg", ".m4a"}


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def git_sha() -> str | None:
    try:
        return subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True, stderr=subprocess.DEVNULL).strip()
    except (OSError, subprocess.CalledProcessError):
        return None


def normal_path(value: str) -> str:
    return Path(value).as_posix().lower()


def physical_inventory() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    split = pd.read_csv(MANIFEST_DIR / "dataset_split.csv")
    unknown = pd.read_csv(MANIFEST_DIR / "unknown_open_set_manifest.csv")
    noise = pd.read_csv(MANIFEST_DIR / "itera_noise_manifest.csv")
    references: dict[str, dict[str, object]] = {}
    for _, row in noise.iterrows():
        parts = Path(str(row.filename)).parts
        references[normal_path(str(Path("data/itera_noise") / row.filename))] = {
            "source": "noise", "role": "verified_bird_free" if bool(row.verified_bird_free) else "mixing_pool",
            "species_key": "", "noise_location": parts[0] if len(parts) > 1 else "unclassified",
        }
    for _, row in split.iterrows():
        references[normal_path(row.file_path)] = {"source": "target", "role": row.split_role, "species_key": row.species_key, "noise_location": ""}
    for _, row in unknown.iterrows():
        category = str(row.source_category)
        references[normal_path(row.file_path)] = {
            "source": "noise_e3_reserved" if category == "itera_environmental_noise" else "unknown_bird",
            "role": row.split_role, "species_key": row.source_label,
            "noise_location": Path(str(row.file_path)).parts[2] if category == "itera_environmental_noise" and len(Path(str(row.file_path)).parts) > 3 else "",
        }

    roots = ("data/xeno_canto_indonesia", "data/xeno_canto_indonesia_unknowns", "data/itera_noise")
    audio_files = [p for root in roots for p in (PROJECT_ROOT / root).rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_SUFFIXES]
    records: list[dict[str, object]] = []
    for number, path in enumerate(sorted(audio_files), start=1):
        relative = path.relative_to(PROJECT_ROOT).as_posix()
        reference = references.get(normal_path(relative), {"source": "unreferenced", "role": "", "species_key": "", "noise_location": ""})
        try:
            info = sf.info(path)
            values = {"container": info.format, "subtype": info.subtype, "sample_rate_hz": int(info.samplerate), "channels": int(info.channels), "duration_seconds": float(info.duration), "decode_error": ""}
        except RuntimeError as error:
            values = {"container": "UNREADABLE_BY_SOUNDFILE", "subtype": "", "sample_rate_hz": np.nan, "channels": np.nan, "duration_seconds": np.nan, "decode_error": str(error)}
        expected = path.suffix.removeprefix(".").upper()
        mismatch = values["container"] not in {expected, "MPEG" if expected == "MP3" else expected}
        records.append({"file_path": relative, "extension": path.suffix.lower(), "extension_container_mismatch": mismatch, **reference, **values})
        if number % 250 == 0 or number == len(audio_files):
            print(f"EDA physical audio: {number}/{len(audio_files)}", flush=True)
    return pd.DataFrame(records), split, unknown, noise


def make_plots(inventory: pd.DataFrame, split: pd.DataFrame, figures_dir: Path) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    split.groupby("species_key").size().sort_values().plot.barh(ax=axes[0, 0], color="#356f9b")
    axes[0, 0].set(title="Active target recordings by species", xlabel="recordings")
    inventory.groupby("source")["duration_seconds"].median().sort_values().plot.bar(ax=axes[0, 1], color="#5a9d6f")
    axes[0, 1].set(title="Median duration by source", ylabel="seconds")
    inventory["sample_rate_hz"].dropna().astype(int).value_counts().sort_index().plot.bar(ax=axes[1, 0], color="#a36c3e")
    axes[1, 0].set(title="Physical sample-rate distribution", xlabel="Hz")
    noise = inventory.loc[inventory["source"].eq("noise")]
    noise["noise_location"].replace("", "unclassified").value_counts().head(12).plot.bar(ax=axes[1, 1], color="#86589c")
    axes[1, 1].set(title="ITERA noise recordings by folder/location", xlabel="location")
    fig.suptitle("Formal EDA: active Indonesian target, unknown buffer, and ITERA noise")
    fig.tight_layout()
    fig.savefig(figures_dir / "eda_overview.png", dpi=180)
    plt.close(fig)


def main() -> Path:
    now = datetime.now(timezone.utc)
    output_dir = OUTPUT_ROOT / f"EDA_{now.strftime('%Y%m%d_%H%M%S')}"
    output_dir.mkdir(parents=True, exist_ok=False)
    tables_dir = output_dir / "tables"
    figures_dir = output_dir / "figures"
    tables_dir.mkdir()
    figures_dir.mkdir()
    inventory, split, unknown, noise = physical_inventory()
    inventory.to_csv(tables_dir / "physical_audio_inventory.csv", index=False)
    grouped = inventory.groupby(["source", "role"], dropna=False).size().rename("count").reset_index()
    durations = inventory.groupby("source")["duration_seconds"].agg(["min", "median", "max"]).reset_index()
    grouped.merge(durations, on="source", how="left").to_csv(tables_dir / "summary_by_source.csv", index=False)
    species = split.groupby(["species_key", "scientific_name", "common_name"], dropna=False).agg(recordings=("recording_id", "count"), recordists=("author", "nunique"), median_rating=("rating", "median")).reset_index().sort_values("species_key")
    species.to_csv(tables_dir / "target_species_summary.csv", index=False)
    inventory.loc[inventory.source.eq("noise")].groupby("noise_location").size().rename("recordings").reset_index().sort_values("recordings", ascending=False).to_csv(tables_dir / "noise_location_summary.csv", index=False)
    make_plots(inventory, split, figures_dir)
    metadata = {
        "artifact_schema": "eda_physical_inventory_v1", "status": "complete", "timestamp_utc": now.isoformat(), "git_sha": git_sha(),
        "manifest_sha256": {name: sha256_file(MANIFEST_DIR / name) for name in ("dataset_split.csv", "unknown_open_set_manifest.csv", "itera_noise_manifest.csv")},
        "physical_audio_count": int(len(inventory)), "manifest_target_count": int(len(split)),
        "manifest_open_set_rows": int(len(unknown)),
        "manifest_unknown_bird_rows": int((unknown.source_category == "non_target_species").sum()),
        "manifest_e3_reserved_noise_rows": int((unknown.source_category == "itera_environmental_noise").sum()),
        "manifest_noise_count": int(len(noise)),
        "soundfile_decode_errors": int(inventory.decode_error.ne("").sum()), "extension_container_mismatches": int(inventory.extension_container_mismatch.sum()), "unreferenced_physical_audio": int(inventory.source.eq("unreferenced").sum()),
    }
    (output_dir / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
    (output_dir / "README.md").write_text(
        "# EDA — Inventaris audio fisik\n\n"
        "- `tables/physical_audio_inventory.csv`: detail setiap audio.\n"
        "- `tables/summary_by_source.csv`: ringkasan sumber.\n"
        "- `tables/target_species_summary.csv`: ringkasan spesies.\n"
        "- `tables/noise_location_summary.csv`: lokasi noise.\n"
        "- `figures/eda_overview.png`: gambar EDA.\n",
        encoding="utf-8",
    )
    print(f"EDA complete: {output_dir}")
    return output_dir

print("Kode run_eda_formal.py telah dimuat di notebook.")


Kode run_eda_formal.py telah dimuat di notebook.


In [3]:
run = ROOT / "results/reproductions/EDA_20261011_202714"
assert run.is_dir(), run
inventory = pd.read_csv(run / "tables/physical_audio_inventory.csv")
split = pd.read_csv(ROOT / "data/manifests/dataset_split.csv")
unknown = pd.read_csv(ROOT / "data/manifests/unknown_open_set_manifest.csv")
noise_manifest = pd.read_csv(ROOT / "data/manifests/itera_noise_manifest.csv")
print(json.dumps(json.loads((run / "metadata.json").read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
display(pd.read_csv(run / "tables/summary_by_source.csv"))
display(pd.read_csv(run / "tables/target_species_summary.csv"))
display(pd.read_csv(run / "tables/noise_location_summary.csv"))


{
  "artifact_schema": "eda_physical_inventory_v1",
  "status": "complete",
  "timestamp_utc": "2026-10-11T20:27:14.798648+00:00",
  "git_sha": "930f6f2ea381221ba29fa09227e86d3d3df1eb33",
  "manifest_sha256": {
    "dataset_split.csv": "7808cdf057f135aaa193f47e27415318edf46c862c09dee4c7394bc5760e8ed8",
    "unknown_open_set_manifest.csv": "a7cd0b26e87e1eadcf8cc4fe13257a5c9e3cf80d740d9ff0016e46e5479f3266",
    "itera_noise_manifest.csv": "a4138bd13e437801d18dc7afde52e0dab50aa5d56ac27091ab0a4ba68fb6f72d"
  },
  "physical_audio_count": 3396,
  "manifest_target_count": 1216,
  "manifest_open_set_rows": 543,
  "manifest_unknown_bird_rows": 272,
  "manifest_e3_reserved_noise_rows": 271,
  "manifest_noise_count": 1799,
  "soundfile_decode_errors": 0,
  "extension_container_mismatches": 101,
  "unreferenced_physical_audio": 109
}


,source,role,count,min,median,max
0,noise,verified_bird_free,1528,0.736000,55.000000,55.000000
1,noise_e3_reserved,unknown_calibration,116,2.280000,55.000000,55.000000
2,noise_e3_reserved,unknown_test,155,2.280000,55.000000,55.000000
3,target,calibration,232,1.206485,33.416899,693.479683
4,target,gallery,673,1.206485,33.416899,693.479683
5,target,query_clean,311,1.206485,33.416899,693.479683
6,unknown_bird,unknown_calibration,117,3.934853,37.611453,432.962646
7,unknown_bird,unknown_test,155,3.934853,37.611453,432.962646
8,unreferenced,NaN,109,4.587792,41.960816,508.411146


,species_key,scientific_name,common_name,recordings,recordists,median_rating
0,ashtai1,Orthotomus ruficeps,Ashy Tailorbird,30,15,4.0
1,colkin1,Todiramphus chloris,Collared Kingfisher,40,14,4.0
2,comior1,Aegithina tiphia,Common Iora,39,18,4.0
3,crseag1,Spilornis cheela,Crested Serpent Eagle,35,15,4.0
4,ghcfly1,Culicicapa ceylonensis,Grey-headed Canary-flycatcher,38,19,4.0
5,gpipig1,Ducula aenea,Green Imperial Pigeon,45,19,4.0
6,hacdro1,Dicrurus hottentottus,Hair-crested Drongo,91,19,4.0
7,hoopit1,Pitta sordida,Hooded Pitta,62,22,4.0
8,horbab1,Malacocincla sepiaria,Horsfield's Babbler,68,32,5.0
9,latnig1,Caprimulgus macrurus,Large-tailed Nightjar,47,26,4.0


,noise_location,recordings
0,Kebun Raya Amplitudo,159
1,Masjid At-tanwir Amplitudo,158
2,Sekitar GKU 1 Amplitudo,156
3,Embung F ITERA Amplitudo,156
4,Masjid At-tanwir Frequency,155
5,Gedung F Amplitudo,150
6,Kebun Raya Frequency,150
7,Embung F ITERA Frequency,149
8,Gedung F Frequency,149
9,Sekitar GKU 1 Frequency,146


In [4]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
split.groupby("species_key").size().sort_values().plot.barh(ax=axes[0, 0], color="#356f9b")
axes[0, 0].set_title("Jumlah rekaman target per spesies")
axes[0, 0].set_xlabel("rekaman")
inventory.groupby("source")["duration_seconds"].median().sort_values().plot.bar(ax=axes[0, 1], color="#5a9d6f")
axes[0, 1].set_title("Median durasi menurut sumber")
axes[0, 1].set_ylabel("detik")
inventory["sample_rate_hz"].dropna().astype(int).value_counts().sort_index().plot.bar(ax=axes[1, 0], color="#a36c3e")
axes[1, 0].set_title("Distribusi sampling rate fisik")
axes[1, 0].set_xlabel("Hz")
noise_locations = inventory.loc[inventory.source.eq("noise"), "noise_location"].replace("", "unclassified").value_counts().head(12)
noise_locations.plot.bar(ax=axes[1, 1], color="#86589c")
axes[1, 1].set_title("Rekaman noise ITERA menurut lokasi/folder")
axes[1, 1].tick_params(axis="x", rotation=35)
fig.suptitle("EDA fisik: target Indonesia, unknown buffer, dan noise ITERA")
fig.tight_layout()
display(fig)
plt.close(fig)


<Figure size 1500x1000 with 4 Axes>

In [5]:
examples = [
    ("Query XC42472", ROOT / "data/xeno_canto_indonesia/savnig1/XC42472.mp3"),
    ("Gallery relevan XC120964", ROOT / "data/xeno_canto_indonesia/savnig1/XC120964.mp3"),
    ("Noise ITERA", ROOT / "data/itera_noise" / str(noise_manifest.iloc[0]["filename"])),
]
fig, axes = plt.subplots(len(examples), 2, figsize=(15, 10))
for row, (label, path) in enumerate(examples):
    audio, sr = librosa.load(str(path), sr=32000, mono=True, duration=5.0)
    time = np.arange(len(audio)) / sr
    axes[row, 0].plot(time, audio, linewidth=0.45, color="#356f9b")
    axes[row, 0].set(title=f"Waveform — {label}", xlabel="detik", ylabel="amplitudo")
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_fft=1024, hop_length=320, n_mels=80, fmax=16000)
    image = axes[row, 1].imshow(librosa.power_to_db(mel, ref=np.max), origin="lower", aspect="auto", cmap="magma", extent=[0, len(audio) / sr, 0, 16000])
    axes[row, 1].set(title=f"Mel-spectrogram — {label}", xlabel="detik", ylabel="Hz")
    fig.colorbar(image, ax=axes[row, 1], label="dB")
fig.tight_layout()
display(fig)
plt.close(fig)
print("Figur dibuat langsung oleh cell ini dari audio fisik dan tidak disimpan/ditimpa ke results.")


c:\Users\Fabio\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Figure size 1500x1000 with 9 Axes>

Figur dibuat langsung oleh cell ini dari audio fisik dan tidak disimpan/ditimpa ke results.


In [6]:
format_issues = inventory.loc[inventory.extension_container_mismatch, ["file_path", "extension", "container", "sample_rate_hz", "duration_seconds"]]
display(format_issues.head(50))
print("Jumlah mismatch ekstensi/container:", len(format_issues))
print("Inventaris lengkap:", run / "tables/physical_audio_inventory.csv")


,file_path,extension,container,sample_rate_hz,duration_seconds
1799,data/xeno_canto_indonesia/ashtai1/XC1003648.mp3,.mp3,WAV,48000,35.536563
1801,data/xeno_canto_indonesia/ashtai1/XC1157504.mp3,.mp3,WAV,44100,30.011837
1829,data/xeno_canto_indonesia/colkin1/XC1003705.mp3,.mp3,WAV,48000,4.477458
1830,data/xeno_canto_indonesia/colkin1/XC1004275.mp3,.mp3,WAV,48000,16.840729
1831,data/xeno_canto_indonesia/colkin1/XC1059449.mp3,.mp3,WAV,192000,6.176469
1865,data/xeno_canto_indonesia/colkin1/XC825797.mp3,.mp3,WAV,48000,64.152083
1869,data/xeno_canto_indonesia/comior1/XC1003838.mp3,.mp3,WAV,48000,11.434646
1870,data/xeno_canto_indonesia/comior1/XC1059451.mp3,.mp3,WAV,192000,7.122260
1871,data/xeno_canto_indonesia/comior1/XC1082290.mp3,.mp3,WAV,44100,32.171247
1907,data/xeno_canto_indonesia/comior1/XC938454.mp3,.mp3,WAV,44100,46.567143


Jumlah mismatch ekstensi/container: 101
Inventaris lengkap: D:\FILE AND TASK\TA\results\reproductions\EDA_20261011_202714\tables\physical_audio_inventory.csv


In [ ]:
# Gambar EDA yang telah disimpan untuk laporan: tampilkan tanpa menghitung ulang audio.
report_figures = ROOT / "results" / "figures"
report_tables = ROOT / "results" / "tables"
figure_names = [
    "02_eda_overview.png",
    "03_eda_bird_waveform_spectrogram_examples.png",
    "04_eda_duration_sampling_rate.png",
    "05_eda_itera_noise_spectrogram_profiles.png",
    "06_eda_noise_locations.png",
    "07_eda_species_distribution.png",
]
print("Folder gambar laporan:", report_figures)
print("Folder tabel laporan:", report_tables)
for name in figure_names:
    figure = report_figures / name
    assert figure.is_file(), figure
    print(name)
    display(Image(filename=str(figure)))

for name in [
    "04_eda_summary_by_source.csv",
    "05_eda_target_species_summary.csv",
    "06_eda_noise_location_summary.csv",
    "08_eda_visual_summary.csv",
]:
    table = report_tables / name
    assert table.is_file(), table
    print(name)
    display(pd.read_csv(table))

print("Inventaris fisik lengkap:", report_tables / "07_eda_physical_audio_inventory.csv")
